# Pandas Essentials for Data Engineers
## Assignment 1 of 2: Data Wrangling Fundamentals with Olist

Practical Data Engineering Course

---


## Introduction

Over the last two sessions you learned the building blocks of Pandas: loading CSVs, exploring a DataFrame, filtering rows, creating new columns, parsing dates, cleaning dirty data, and more. That training used a small generated dataset so we could focus on the mechanics.

This is the first of two assignments that move you into real life. Assignment 1 focuses on single-table wrangling: loading data, exploring it, selecting and filtering, computing new columns, working with dates, and cleaning dirty tables one at a time.

Assignment 2 will focus on integration: joining multiple tables, aggregating, reshaping, and building a final ETL pipeline that ends in a SQLite database.

Your goal in Assignment 1 is not only to write code that runs, but to make the decisions a real data engineer makes every day: why keep this row, why fill this null with the median, why cast this column to datetime.


## The Dataset

Primary dataset: Brazilian E-Commerce Public Dataset by Olist.

Source on Kaggle: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

This dataset contains roughly 100,000 real orders from 2016 to 2018 placed on Olist, a Brazilian marketplace platform. It is split into nine CSV files. In this assignment you will use the following:

- olist_orders_dataset.csv
- olist_order_items_dataset.csv
- olist_products_dataset.csv
- olist_customers_dataset.csv
- olist_order_payments_dataset.csv
- product_category_name_translation.csv

## Business Scenario

You have just joined the analytics engineering team at Olist. The raw CSV exports from the operational database are sitting in a shared folder waiting for you. Before any analytics happens, someone needs to profile the data, understand it, filter it down to what matters, enrich it with helpful columns, and clean it up.

That someone is you. In this assignment you will produce a set of clean, enriched tables that the next assignment will use to build the pipeline.


## What to Submit

- This notebook, renamed to `olist_assignment_1_<your_name>.ipynb`, with your code and markdown notes.
- A short written reflection at the end, about half a page, discussing the most interesting observations you made about data quality.
- Clean CSV exports of the cleaned orders, items, and products tables.


## Setup

Run this cell before starting. Place the CSV files in a folder called `raw/` next to this notebook.


In [721]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Adjust this path if your files are in a different location
DATA_DIR = 'raw'


---
# Level 1: Foundations

Before writing any transformation, you must know the shape of your data. In this level you just look, observe, and take notes. No changes to the data yet.


## Question 1: Loading the three core files

Scenario:

On your first day you are told the data lives in CSVs in a folder called raw. Load the orders, order_items, and products files into three separate DataFrames.

Tasks:
- Read each CSV into a DataFrame using the appropriate loading function.
- Print the shape of each DataFrame.
- Print the column names of each DataFrame.

Hints:

```python
pd.read_csv('raw/olist_orders_dataset.csv')
.shape returns a tuple (rows, columns)
.columns.tolist() gives a clean list
```


In [722]:
# Write your solution for Question 1 here
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_orders.shape
print(f"\n olist orders dataset loaded: {df_orders.shape[0]} rows, {df_orders.shape[1]} columns")
df_orders.columns.tolist()


 olist orders dataset loaded: 99441 rows, 8 columns


['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [723]:
df_items = pd.read_csv('olist_order_items_dataset.csv')
df_items.shape
print(f"\n olist orders items dataset loaded: {df_items.shape[0]} rows, {df_items.shape[1]} columns")
df_items.columns.tolist()


 olist orders items dataset loaded: 112650 rows, 7 columns


['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value']

In [724]:
df_category = pd.read_csv('product_category_name_translation.csv')
df_category.shape
print(f"\n product category name loaded: {df_category.shape[0]} rows, {df_category.shape[1]} columns")
df_category.columns.tolist()


 product category name loaded: 71 rows, 2 columns


['product_category_name', 'product_category_name_english']

In [725]:
df_products = pd.read_csv('olist_products_dataset.csv')
df_products.shape
print(f"\n olist products dataset loaded: {df_products.shape[0]} rows, {df_products.shape[1]} columns")
df_products.columns.tolist()



 olist products dataset loaded: 32951 rows, 9 columns


['product_id',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

In [726]:
df_payments = pd.read_csv('olist_order_payments_dataset.csv')
df_payments.shape
print(f"\n olist orders payments dataset loaded: {df_payments.shape[0]} rows, {df_payments.shape[1]} columns")
df_payments.columns.tolist()


 olist orders payments dataset loaded: 103886 rows, 5 columns


['order_id',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value']

## Question 2: First look at the orders table

Scenario:

Your manager asks: what do the first few orders look like, and what is in the last batch of rows?

Tasks:
- Show the first 7 rows of orders.
- Show the last 3 rows of orders.
- Check the data types of each column.

Hints:

```python
.head(n) and .tail(n)
.dtypes
```


In [727]:
# Write your solution for Question 2 here
df_orders.head(7)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00


In [728]:
df_orders.tail(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00
99440,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-03-08 20:57:30,2018-03-09 11:20:28,2018-03-09 22:11:59,2018-03-16 13:08:30,2018-04-03 00:00:00


In [729]:

df_orders.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

## Question 3: Full profile

Scenario:

You will need a one-shot summary that shows nulls and dtypes together. You will also want a numeric summary to see ranges.

Tasks:
- Use one method that returns column names, non-null counts, and dtypes in a single view.
- Use another method that returns statistical summary (count, mean, std, min, percentiles, max) for numeric columns.

Hints:

```python
.info() shows nulls and dtypes
.describe() for numeric summary, .describe(include='object') for text columns
```


In [730]:
# Write your solution for Question 3 here
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [731]:
df_orders.describe()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-08-02 12:05:26,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-08 19:36:48,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


## Question 4: Unique counts

Scenario:

Marketing asks how many unique customers placed orders and how many distinct order statuses exist.

Tasks:
- Count unique customer_id values in orders.
- Count how many times each order_status appears.

Hints:

```python
.nunique() for a count of distinct values
.value_counts() for a breakdown
```


In [733]:
# Write your solution for Question 4 here
df_orders["customer_id"].nunique()

99441

In [734]:
df_orders["order_status"].value_counts()


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## Question 5: Manual DataFrame

Scenario:

To test your aggregations later, build a tiny DataFrame by hand with three products and their cost.

Tasks:
- Create a DataFrame with columns product, category, and cost from a Python dictionary.

Hints:

```python
pd.DataFrame({'product': [...], 'category': [...], 'cost': [...]})
```


In [735]:
# Write your solution for Question 5 here
sample = pd.DataFrame({
    'product': ['Laptop', 'Mouse', 'MS_Office'],
    'category':   ['Device', "Accessories", 'Software'],
    'cost':   [500, 20, 150]
})
sample

,product,category,cost
0,Laptop,Device,500
1,Mouse,Accessories,20
2,MS_Office,Software,150


---
# Level 2: Selecting and Filtering

Real questions from real teams rarely need the whole table. You have to slice, filter, and return only what the stakeholder needs.


## Question 6: Pick the columns the CEO cares about

Scenario:

The CEO wants a one-page view of orders and does not care about dates. Give her a view with only order_id, customer_id, order_status, and order_purchase_timestamp.

Tasks:
- Return a DataFrame with only those four columns.
- Save it as df_ceo.

Hints:

```python
df[['col1', 'col2', 'col3']] returns a DataFrame
Single brackets df['col'] returns a Series
```


In [736]:
# Write your solution for Question 6 here
df_ceo = df_orders[['order_id','customer_id', 'order_status','order_purchase_timestamp']]
df_ceo

,order_id,customer_id,order_status,order_purchase_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39
...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27


## Question 7: Delivered orders only

Scenario:

The logistics team only wants orders that were successfully delivered.

Tasks:
- Filter the orders DataFrame to keep only rows where order_status equals delivered.
- Report how many rows remain and what percentage of the total that is.

Hints:

```python
df[df['col_name'] == 'value']
len(filtered) / len(df)
```


In [737]:
# Write your solution for Question 7 here
delivered = df_orders[df_orders['order_status'] == 'delivered']
delivered.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [738]:
print (f"Total orders:{len(df_orders['order_status'])}")

Total orders:99441


In [739]:
# Method 1
df_orders[df_orders['order_status'] == 'delivered'].value_counts(dropna=False).sum()


np.int64(96478)

In [740]:
# Method 2
print (f"Rows with delivered status: {len(df_orders[df_orders['order_status'] == 'delivered'])}")

Rows with delivered status: 96478


In [741]:
print(f"Percentage of delivered product: {len(delivered) / len(df_orders) * 100} %")

Percentage of delivered product: 97.02034372140264 %


## Question 8: Problem orders

Scenario:

Customer support is firefighting and needs a list of orders that are canceled or unavailable.

Tasks:
- Filter using a list of statuses instead of multiple OR conditions.
- Count them.

Hints:

```python
df['col_name'].isin(['col_name', ....])
use ~ in front to invert the mask (not)
```


In [742]:
# Write your solution for Question 8 here
bad_orders = df_orders[~df_orders['order_status'].isin(['delivered','shipped','invoiced','processing','created','approved'])]
len(bad_orders)


1234

## Question 9: Label based versus position based selection

Scenario:

Practice both .loc and .iloc on the orders table.

Tasks:
- Use .loc to pick rows where order_status is shipped and return only the columns order_id and order_purchase_timestamp.
- Use .iloc to return the first 5 rows and the first 3 columns regardless of their names.

Hints:

```python
df.loc[mask, ['col1', 'col2']]
df.iloc[start:end, start:end]
```


In [743]:
# Write your solution for Question 9 here
df_orders.loc[df_orders['order_status'] == "shipped", ['order_id', 'order_purchase_timestamp']].head()

,order_id,order_purchase_timestamp
44,ee64d42b8cf066f35eac1cf57de1aa85,2018-06-04 16:44:48
154,6942b8da583c2f9957e990d028607019,2018-01-10 11:33:07
162,36530871a5e80138db53bcfd8a104d90,2017-05-09 11:48:37
231,4d630f57194f5aba1a3d12ce23e71cd9,2017-11-17 19:53:21
299,3b4ad687e7e5190db827e1ae5a8989dd,2018-06-28 12:52:15


In [744]:
df_orders.iloc[0:5, 0:3]

,order_id,customer_id,order_status
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered


## Question 10: Readable complex filter

Scenario:

Write the same filter twice: once with bracket boolean masks, once with the .query method. Compare.

Tasks:
- Find order_items rows where price is between 100 and 300 and freight_value is greater than 20.

Hints:

```python
df[(df['col_name'].between(vlaue,value)) & (df['col_name'] > value)]
df.query('write your query here ')
```


In [745]:
# Write your solution for Question 10 here
df_items[(df_items['price'].between(100,300)) & (df_items['freight_value'] > 20)]


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
22,000f25f4d72195062c040b12dce9a18a,1,1c05e0964302b6cf68ca0d15f326c6ba,7c67e1448b00f6e969d365cea6b010ab,2018-03-21 11:10:11,119.99,44.40
26,0011d82c4b53e22e84023405fb467e57,1,c389f712c4b4510bc997cee93e8b1a28,bfd27a966d91cfaafdb25d076585f0da,2018-01-29 21:51:25,289.00,26.33
27,00125cb692d04887809806618a2a145f,1,1c0c0093a48f13ba70d0c6b0a9157cb7,41b39e28db005d9731d9d485a83b4c38,2017-03-29 13:05:42,109.90,25.51
59,00254baeb6c932b0a8aeead91fbd02b5,1,18901878788fec7ddc55e64d1ace8187,e8b4225284fbb02d16f200513f1f395d,2018-05-14 22:14:46,149.90,43.11
62,002611a77fe03d076285fd4ca95db77c,1,fe077ec80df6b4ee60bb4498d5ab1962,87142160b41353c4e5fca2360caf6f92,2018-03-07 02:49:09,135.00,21.75
...,...,...,...,...,...,...,...
112634,fff8287bbae429a99bb7e8c21d151c41,1,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-27 12:29:22,180.00,48.14
112635,fff8287bbae429a99bb7e8c21d151c41,2,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-27 12:29:22,180.00,48.14
112637,fffa82886406ccf10c7b4e35c4ff2788,1,bbe7651fef80287a816ead73f065fc4b,8f2ce03f928b567e3d56181ae20ae952,2017-12-22 17:31:42,229.90,44.02
112644,fffbee3b5462987e66fb49b1c5411df2,1,6f0169f259bb0ff432bfff7d829b9946,213b25e6f54661939f11710a6fddb871,2018-06-28 09:58:03,119.85,20.03


In [746]:
df_items.query('100 <= price <= 300 and freight_value > 20 ')

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
22,000f25f4d72195062c040b12dce9a18a,1,1c05e0964302b6cf68ca0d15f326c6ba,7c67e1448b00f6e969d365cea6b010ab,2018-03-21 11:10:11,119.99,44.40
26,0011d82c4b53e22e84023405fb467e57,1,c389f712c4b4510bc997cee93e8b1a28,bfd27a966d91cfaafdb25d076585f0da,2018-01-29 21:51:25,289.00,26.33
27,00125cb692d04887809806618a2a145f,1,1c0c0093a48f13ba70d0c6b0a9157cb7,41b39e28db005d9731d9d485a83b4c38,2017-03-29 13:05:42,109.90,25.51
59,00254baeb6c932b0a8aeead91fbd02b5,1,18901878788fec7ddc55e64d1ace8187,e8b4225284fbb02d16f200513f1f395d,2018-05-14 22:14:46,149.90,43.11
62,002611a77fe03d076285fd4ca95db77c,1,fe077ec80df6b4ee60bb4498d5ab1962,87142160b41353c4e5fca2360caf6f92,2018-03-07 02:49:09,135.00,21.75
...,...,...,...,...,...,...,...
112634,fff8287bbae429a99bb7e8c21d151c41,1,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-27 12:29:22,180.00,48.14
112635,fff8287bbae429a99bb7e8c21d151c41,2,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-27 12:29:22,180.00,48.14
112637,fffa82886406ccf10c7b4e35c4ff2788,1,bbe7651fef80287a816ead73f065fc4b,8f2ce03f928b567e3d56181ae20ae952,2017-12-22 17:31:42,229.90,44.02
112644,fffbee3b5462987e66fb49b1c5411df2,1,6f0169f259bb0ff432bfff7d829b9946,213b25e6f54661939f11710a6fddb871,2018-06-28 09:58:03,119.85,20.03


In [747]:
df_items.query('price.between (100,300) and freight_value > 20 ')

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
22,000f25f4d72195062c040b12dce9a18a,1,1c05e0964302b6cf68ca0d15f326c6ba,7c67e1448b00f6e969d365cea6b010ab,2018-03-21 11:10:11,119.99,44.40
26,0011d82c4b53e22e84023405fb467e57,1,c389f712c4b4510bc997cee93e8b1a28,bfd27a966d91cfaafdb25d076585f0da,2018-01-29 21:51:25,289.00,26.33
27,00125cb692d04887809806618a2a145f,1,1c0c0093a48f13ba70d0c6b0a9157cb7,41b39e28db005d9731d9d485a83b4c38,2017-03-29 13:05:42,109.90,25.51
59,00254baeb6c932b0a8aeead91fbd02b5,1,18901878788fec7ddc55e64d1ace8187,e8b4225284fbb02d16f200513f1f395d,2018-05-14 22:14:46,149.90,43.11
62,002611a77fe03d076285fd4ca95db77c,1,fe077ec80df6b4ee60bb4498d5ab1962,87142160b41353c4e5fca2360caf6f92,2018-03-07 02:49:09,135.00,21.75
...,...,...,...,...,...,...,...
112634,fff8287bbae429a99bb7e8c21d151c41,1,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-27 12:29:22,180.00,48.14
112635,fff8287bbae429a99bb7e8c21d151c41,2,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-27 12:29:22,180.00,48.14
112637,fffa82886406ccf10c7b4e35c4ff2788,1,bbe7651fef80287a816ead73f065fc4b,8f2ce03f928b567e3d56181ae20ae952,2017-12-22 17:31:42,229.90,44.02
112644,fffbee3b5462987e66fb49b1c5411df2,1,6f0169f259bb0ff432bfff7d829b9946,213b25e6f54661939f11710a6fddb871,2018-06-28 09:58:03,119.85,20.03


---
# Level 3: Creating and Computing Columns

A data engineer adds columns the business can actually use. Avoid Python loops, use vectorized operations, and keep the transformations explicit so a teammate can read them.


## Question 11: Total item cost

Scenario:

The finance team wants a total_item_cost column that represents price plus freight, per row in order_items.

Tasks:
- Create a new column on df_items that equals price plus freight_value.
- Round the result to two decimals.

Hints:

```python
df['col_name'] = df['col_name'] *-/+ df['col_name'].round(n)
```


In [748]:
# Write your solution for Question 11 here
df_items['total_item_cost'] = (df_items['price'] + df_items['freight_value']).round(2)
df_items[['price', 'freight_value', 'total_item_cost']].head()

,price,freight_value,total_item_cost
0,58.90,13.29,72.19
1,239.90,19.93,259.83
2,199.00,17.87,216.87
3,12.99,12.79,25.78
4,199.90,18.14,218.04


## Question 12: Expensive flag with np.where

Scenario:

Marketing wants a yes or no column that flags items considered expensive, using 300 as the cutoff.

Tasks:
- Create an is_expensive column that is 'Yes' when price is greater than 300, and 'No' otherwise.

Hints:

```python
np.where(condition, value_if_true, value_if_false)
```


In [749]:
# Write your solution for Question 12 here
df_items['is_expensive'] = np.where(df_items['price'] > 300, 'Yes' , 'No')
df_items[['price', 'is_expensive']].head(11)

,price,is_expensive
0,58.90,No
1,239.90,No
2,199.00,No
3,12.99,No
4,199.90,No
5,21.90,No
6,19.90,No
7,810.00,Yes
8,145.95,No
9,53.99,No


## Question 13: Price tiers with pd.cut

Scenario:

Categorize every item into a tier that finance can group by: Budget, Mid, Premium, Luxury.

Tasks:
- Use bin edges of your choice to produce four labeled tiers.
- Verify the distribution with value_counts.

Hints:

```python
pd.cut(df['col_name'], bins=[0, 50, 150, 400, 10000], labels=['l1','l2',....])
```


In [750]:
# Write your solution for Question 13 here
df_items['price_tier'] = pd.cut(df_items['price'],
                           bins=[0, 50, 150, 400, 1000],
                           labels=['Budget','Mid','Premium','Luxury'])
df_items[['order_id','price','price_tier']].head(10)


,order_id,price,price_tier
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,Mid
1,00018f77f2f0320c557190d7a144bdd3,239.90,Premium
2,000229ec398224ef6ca0657da4fc703e,199.00,Premium
3,00024acbcdf0a6daa1e931b038114c75,12.99,Budget
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,Premium
5,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,Budget
6,00054e8431b9d7675808bcb819fb4a32,19.90,Budget
7,000576fe39319847cbb9d288c5617fa6,810.00,Luxury
8,0005a1a1728c9d785b8e2b08b904576c,145.95,Mid
9,0005f50442cb953dcd1d21e1fb923495,53.99,Mid


In [751]:
df_items['price_tier'].value_counts()

price_tier
Mid        50878
Budget     39317
Premium    18216
Luxury      3395
Name: count, dtype: int64

## Question 14: Custom function with apply

Scenario:

Logistics has a rule: any item heavier than 5 kg is Heavy, between 1 and 5 kg is Medium, otherwise Light. Apply this rule across the products table.

Tasks:
- Write a Python function that takes a single weight and returns the correct label.
- Apply that function to product_weight_g, remembering the unit is grams.

Hints:

```python
def classify_weight(w): ...
df['col_name'] = df['col_name'].apply(classify_weight)
Handle null weights inside your function to avoid errors
```


In [752]:
# Write your solution for Question 14 here
def classify_weight(w):
    if w > 5: return 'Heavy'
    elif 1 <= w <= 5: return 'Medium'
    else: return 'Light'

df_products['weight_label'] = df_products['product_weight_g'].apply(classify_weight)
df_products[['product_id','product_weight_g','weight_label']].head(11)


,product_id,product_weight_g,weight_label
0,1e9e8ef04dbcff4541ed26657ea517e5,225.0,Heavy
1,3aa071139cb16b67ca9e5dea641aaa2f,1000.0,Heavy
2,96bd76ec8810374ed1b65e291975717f,154.0,Heavy
3,cef67bcfe19066a932b7673e239eb23d,371.0,Heavy
4,9dc1a7de274444849c219cff195d0b71,625.0,Heavy
5,41d3672d4792049fa1779bb35283ed13,200.0,Heavy
6,732bd381ad09e530fe0a5f457d81becb,18350.0,Heavy
7,2548af3e6e77a690cf3eb6368e9ab61e,900.0,Heavy
8,37cc742be07708b53a98702e77a21a02,400.0,Heavy
9,8c92109888e8cdf9d66dc7e463025574,600.0,Heavy


In [753]:
df_products['weight_label'].value_counts()

weight_label
Heavy     32940
Light         6
Medium        5
Name: count, dtype: int64

## Question 15: String operations on categories

Scenario:

Category names in Portuguese are snake_case and hard to read. Produce a clean, title case version.

Tasks:
- Replace underscores with spaces in product_category_name.
- Convert the result to title case.
- Store it as product_category_clean.

Hints:

```python
.str.replace('_', ' ')
.str.title()
```


In [754]:
# Write your solution for Question 15 here
df_products['product_category_clean'] = df_products['product_category_name'].str.replace('_', ' ')
df_products['product_category_clean'].str.title()



0                               Perfumaria
1                                    Artes
2                            Esporte Lazer
3                                    Bebes
4                    Utilidades Domesticas
                       ...                
32946                     Moveis Decoracao
32947    Construcao Ferramentas Iluminacao
32948                      Cama Mesa Banho
32949               Informatica Acessorios
32950                      Cama Mesa Banho
Name: product_category_clean, Length: 32951, dtype: object

## Question 16: Recoding with map

Scenario:

Load the payments file. The payment_type column has values like credit_card, boleto, voucher, debit_card. Translate them to integer codes 1, 2, 3, 4 for a legacy downstream system.

Tasks:
- Define a dictionary that maps each payment type to its integer code.
- Use map to produce payment_code.
- Check that no nulls were introduced.

Hints:

```python
df['col_name'] = df['col_name'].map(payment_map)
A null in the result means an unmapped value; handle it with .fillna(-1) or add the missing keys
```


In [755]:
# Write your solution for Question 16 here
payments_map = {'credit_card': 1, 'boleto': 2, 'voucher': 3, 'debit_card': 4}
df_payments['payment_type_code'] = df_payments['payment_type'].map(payments_map)
df_payments[['payment_type','payment_type_code']].head(10)

,payment_type,payment_type_code
0,credit_card,1.0
1,credit_card,1.0
2,credit_card,1.0
3,credit_card,1.0
4,credit_card,1.0
5,credit_card,1.0
6,credit_card,1.0
7,credit_card,1.0
8,credit_card,1.0
9,boleto,2.0


In [756]:
df_payments['payment_type_code']  = df_payments['payment_type'].fillna(-1)
print(df_payments[['payment_type','payment_type_code']].isnull().sum())

payment_type         0
payment_type_code    0
dtype: int64


---
# Level 4: Working with Dates

In the orders table you have five timestamp columns that describe the life of an order. These are strings until you parse them. After parsing you can compute durations and extract components.


## Question 17: Parse every timestamp column

Scenario:

Convert every date column in the orders table to real datetime values.

Tasks:
- Identify all columns in orders that look like timestamps.
- Convert each one in a loop or individually.
- Confirm the new dtype is datetime64.

Hints:

```python
pd.to_datetime(df['col'], errors='coerce')
errors='coerce' turns invalid strings into NaT instead of raising
```


In [757]:
# Write your solution for Question 17 here
date_cols = [col for col in df_orders.columns 
             if 'date' in col.lower() or 'time' in col.lower()]

for x in date_cols:
    df_orders[x] = pd.to_datetime(df_orders[x], errors='coerce')

In [758]:
 
df_orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                        object
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

## Question 18: Order date parts

Scenario:

Extract useful components from order_purchase_timestamp so BI can group by them later.

Tasks:
- Create columns for year, month, quarter, day_name, and year_month as a period string.

Hints:

```python
.dt.year, .dt.month, .dt.quarter
.dt.day_name()
.dt.to_period('M').astype(str)
```


In [759]:
# Write your solution for Question 18 here
df_orders['order_year']    = df_orders['order_purchase_timestamp'].dt.year           
df_orders['order_month']   = df_orders['order_purchase_timestamp'].dt.month          
df_orders['order_quarter'] = df_orders['order_purchase_timestamp'].dt.quarter       
df_orders['day_name']   = df_orders['order_purchase_timestamp'].dt.day_name()    
df_orders['order_period']  = df_orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

df_orders[['order_year','order_month','order_quarter','day_name','order_period']].head()



,order_year,order_month,order_quarter,day_name,order_period
0,2017,10,4,Monday,2017-10
1,2018,7,3,Tuesday,2018-07
2,2018,8,3,Wednesday,2018-08
3,2017,11,4,Saturday,2017-11
4,2018,2,1,Tuesday,2018-02


## Question 19: Delivery duration

Scenario:

Logistics wants to know, in days, how long between the order purchase and the actual delivery to the customer.

Tasks:
- Compute delivery_days as the difference between order_delivered_customer_date and order_purchase_timestamp.
- Convert the result to integer days.
- Report the mean, median, and maximum.

Hints:

```python
(df['col_name'] - df['col_name']).dt.days
Some rows will be null because the order was not delivered yet
```


In [760]:
# Write your solution for Question 19 here
df_orders['delivery_days'] = (df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']).dt.days
df_orders['delivery_days'] = df_orders['delivery_days'].astype('Int64')
df_orders[['order_delivered_customer_date','order_purchase_timestamp', 'delivery_days']].head()


,order_delivered_customer_date,order_purchase_timestamp,delivery_days
0,2017-10-10 21:25:13,2017-10-02 10:56:33,8
1,2018-08-07 15:27:45,2018-07-24 20:41:37,13
2,2018-08-17 18:06:29,2018-08-08 08:38:49,9
3,2017-12-02 00:28:42,2017-11-18 19:28:06,13
4,2018-02-16 18:17:02,2018-02-13 21:18:39,2


In [761]:
df_orders['delivery_days'].max()


np.int64(209)

In [762]:
df_orders['delivery_days'].mean()


np.float64(12.094085575687217)

In [763]:
df_orders['delivery_days'].median()

np.float64(10.0)

## Question 20: Late deliveries

Scenario:

An order is late when the actual delivery date is after the estimated delivery date. Create a late_delivery boolean column and report the late rate.

Tasks:
- Compare order_delivered_customer_date to order_estimated_delivery_date.
- Compute the percentage of delivered orders that were late.

Hints:

```python
df['col_name'] = df['col_name'] > df['col_name']
Filter to delivered orders first, otherwise nulls will skew the rate
```


In [764]:
# Write your solution for Question 20 here
df_orders['late_delivery'] = df_orders['order_delivered_customer_date'] > df_orders['order_estimated_delivery_date']
df_orders['late_delivery']

0        False
1        False
2        False
3        False
4        False
         ...  
99436    False
99437    False
99438    False
99439    False
99440    False
Name: late_delivery, Length: 99441, dtype: bool

In [765]:
late = len(df_orders[df_orders['late_delivery'] == True])
total_delivery = len(df_orders[df_orders['order_delivered_customer_date'].notna()])
result = (late / total_delivery) * 100
print(f"Late delivery percentage: {round(result, 2)}%")

Late delivery percentage: 8.11%


## Question 21: Slice a time range

Scenario:

The CFO wants a look at orders placed in the last quarter of 2017.

Tasks:
- Return all orders with order_purchase_timestamp between 2017-10-01 and 2017-12-31 inclusive.
- Count them.

Hints:

```python
Compare the datetime column with string dates, pandas will parse them
```


In [766]:
# Write your solution for Question 21 here
q4 = df_orders[(df_orders['order_purchase_timestamp'] >= '2017-10-01') & (df_orders['order_purchase_timestamp'] <= '2017-12-31')]
print(f"Q4 2017 orders: {len(q4)}")

Q4 2017 orders: 17774


---
# Level 5: Cleaning Single Tables

Every answer above is wrong if the data is dirty. In this level you audit and clean the individual tables before any joining happens in Assignment 2.


## Question 22: Null audit on orders

Scenario:

Produce a nulls report for the orders table that every new data engineer should keep as a template.

Tasks:
- For each column in orders, compute the count of nulls and the percentage.
- Sort by percentage descending.
- Return a small DataFrame summary, not a wall of text.

Hints:

```python
df.isnull().sum()
(df.isnull().sum() / len(df) * 100).round(2)
Build a new DataFrame from the two Series using pd.concat along axis=1
```


In [767]:
# Write your solution for Question 22 here
df_orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
order_year                          0
order_month                         0
order_quarter                       0
day_name                            0
order_period                        0
delivery_days                    2965
late_delivery                       0
dtype: int64

In [768]:
(df_orders.isnull().sum()/len(df_orders) * 100).round(2).sort_values(ascending= False)

order_delivered_customer_date    2.98
delivery_days                    2.98
order_delivered_carrier_date     1.79
order_approved_at                0.16
order_status                     0.00
order_id                         0.00
customer_id                      0.00
order_purchase_timestamp         0.00
order_estimated_delivery_date    0.00
order_month                      0.00
order_year                       0.00
order_quarter                    0.00
day_name                         0.00
order_period                     0.00
late_delivery                    0.00
dtype: float64

In [769]:
df_null_value = df_orders.isnull().sum().copy()
df_null_percentage = (df_orders.isnull().sum()/len(df_orders) * 100).round(2).copy()

df_combined = pd.concat([df_null_value, df_null_percentage], ignore_index=False, axis=1)
df_combined.columns = ['missing_value', 'missing_percentage']
view = df_combined.sort_values('missing_percentage',ascending= False)
view


,missing_value,missing_percentage
order_delivered_customer_date,2965,2.98
delivery_days,2965,2.98
order_delivered_carrier_date,1783,1.79
order_approved_at,160,0.16
order_status,0,0.00
order_id,0,0.00
customer_id,0,0.00
order_purchase_timestamp,0,0.00
order_estimated_delivery_date,0,0.00
order_month,0,0.00


## Question 23: Impute or drop

Scenario:

Make decisions for three specific columns in the products table and justify each one in markdown.

Tasks:
- product_weight_g: pick a strategy and apply it.
- product_name_lenght: pick a strategy and apply it.
- product_category_name: pick a strategy and apply it.

Hints:

```python
.fillna(df['col'].median())
.fillna('unknown')
.dropna(subset=['col'])
Dropping rows is safer for timestamps than filling them with a fake date
```


In [776]:
# Write your solution for Question 23 here
df_products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
weight_label                    0
product_category_clean        610
dtype: int64

In [777]:
df_products['product_weight_g']= df_products['product_weight_g'].fillna(df_products['product_weight_g'].median())
df_products['product_name_lenght']= df_products['product_name_lenght'].fillna(df_products['product_name_lenght'].median())
df_products['product_category_name']= df_products['product_category_name'].fillna("unknown")


In [778]:
df_products.isnull().sum()

product_id                      0
product_category_name           0
product_name_lenght             0
product_description_lenght    610
product_photos_qty            610
product_weight_g                0
product_length_cm               2
product_height_cm               2
product_width_cm                2
weight_label                    0
product_category_clean        610
dtype: int64

## Question 24: Drop columns with too many nulls

Scenario:

Drop any columns in the orders table where more than 50 percent of the values are null.

Tasks:
- Use a threshold on dropna with axis=1.
- Print the list of columns removed.

Hints:

```python
df.dropna(axis=1, thresh=int(len(df)*0.5))
Before calling, capture the original column list to compute the difference
```


In [779]:
# Write your solution for Question 24 here
df_orders.isnull().sum() / len(df_orders) * 100  

order_id                         0.000000
customer_id                      0.000000
order_status                     0.000000
order_purchase_timestamp         0.000000
order_approved_at                0.160899
order_delivered_carrier_date     1.793023
order_delivered_customer_date    2.981668
order_estimated_delivery_date    0.000000
order_year                       0.000000
order_month                      0.000000
order_quarter                    0.000000
day_name                         0.000000
order_period                     0.000000
delivery_days                    2.981668
late_delivery                    0.000000
dtype: float64

In [783]:
original_columns = df_orders.columns.tolist()
original_columns 

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'order_year',
 'order_month',
 'order_quarter',
 'day_name',
 'order_period',
 'delivery_days',
 'late_delivery']

In [787]:
df_trimmed = df_orders.dropna(axis=1 ,thresh=int(len(df_orders)*0.5))
removed =[ i for i in original_columns if i not in df_trimmed.columns]
print("Columns that have been removed due to a missing values of 50% or more:", removed)
print("Remaining columns:", df_trimmed.columns.tolist())

Columns that have been removed due to a missing values of 50% or more: []
Remaining columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_year', 'order_month', 'order_quarter', 'day_name', 'order_period', 'delivery_days', 'late_delivery']


## Question 25: Duplicate detection

Scenario:

Check if order_items has duplicate rows. Some duplicates are legitimate because one order can have the same product twice, others are errors. Decide.

Tasks:
- Count fully duplicated rows.
- Count rows that are duplicated on the pair (order_id, product_id).
- Decide whether to drop any of them and explain in markdown.

Hints:

```python
.duplicated().sum()
.duplicated(subset=['col_name',...], keep=False).sum()
```


In [788]:
# Write your solution for Question 25 here
df_items.duplicated().sum()

np.int64(0)

In [789]:
df_items.duplicated(subset=['order_id', 'product_id']).sum()

np.int64(10225)

In [790]:
df_items.drop_duplicates(subset=['order_id', 'product_id'], inplace=True)

In [792]:
df_items.duplicated().sum()

np.int64(0)

## Question 26: Invalid business records

Scenario:

Inspect the order_items numeric columns for impossible values.

Tasks:
- Report the count of rows where price is less than or equal to zero.
- Report the count where freight_value is negative.
- Remove the bad rows and keep a clean copy.

Hints:

```python
(df['col_name'] <= 0).sum()
df = df[df['col_name'] > 0]
```


In [794]:
# Write your solution for Question 26 here
(df_items['price']<=0).sum()

np.int64(0)

In [795]:
(df_items['freight_value']<0).sum()

np.int64(0)

In [798]:
df_clean = df_items[(df_items['price']>0) & (df_items['freight_value']>0)]    
df_clean.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,total_item_cost,is_expensive,price_tier
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19,No,Mid
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83,No,Premium
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87,No,Premium
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78,No,Budget
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04,No,Premium


## Question 27: Type casting and renaming

Scenario:

Finalize dtypes and names for the cleaned orders table.

Tasks:
- Ensure numeric columns are actually numeric.
- Ensure timestamp columns are datetime.
- Rename any column for clarity if needed and document the change.

Hints:

```python
.astype(float), .astype(int)
pd.to_datetime(df['col'])
df.rename(columns={'old': 'new'})
```


In [799]:
# Write your solution for Question 27 here
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  object        
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   order_year                     99441 non-null  int32         
 9   order_month                    99441 non-null  int32         
 10  order_quarter                  99441 non-null  int32         
 11  day_name       

---
# Export Your Cleaned Tables

Before submitting, save the cleaned orders, items, and products tables as CSVs. Assignment 2 will assume these exist (or you can re-run from raw files).


In [775]:
# Example:
# df_orders_clean.to_csv('clean/orders_clean.csv', index=False)
# df_items_clean.to_csv('clean/items_clean.csv', index=False)
# df_products_clean.to_csv('clean/products_clean.csv', index=False)


---
# Reflection

Write about half a page of markdown below describing the three most interesting things you learned about data quality in this assignment. What would you tell a new data engineer to watch out for when they pick up this dataset for the first time?


*Your reflection here.*
